*  DSC670-T301 Advanced Uses of Generative AI (2265-1)
*  8.3 Project Milestone 3 - Build Your First Model
*  Charles Sarratt

# AI-Assisted Experiment Analysis Fine-Tuning

This notebook builds on the prompt experimentation completed in Milestone 2. In that milestone, structured and constraint-based prompting produced the most reliable results for interpreting A/B test outcomes.

For this milestone, the goal is to create a fine-tuned OpenAI model and evaluate whether fine-tuning improves the consistency, formatting, or decision quality of the experiment analysis output.

The model is not being used to perform statistical calculations. Instead, it interprets provided experiment metrics and generates a structured recommendation.

## Environment Setup

This section loads the required libraries and initializes the OpenAI client. The API key is loaded from a local environment file so that the key is not stored directly in the notebook.

This setup follows the same pattern used in the Milestone 2 notebook.

In [1]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("openai.env")
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("OpenAI client initialized:", os.getenv("OPENAI_API_KEY") is not None)

OpenAI client initialized: True


### Setup Result

The OpenAI client initialized successfully, which confirms that the API key was loaded correctly from the local environment file. This allows the rest of the notebook to use the OpenAI API without placing the API key directly in the code.

## Training Dataset Design

The training dataset uses synthetic A/B test scenarios. This is appropriate for this project because the goal is not to train the model on proprietary experiment data, but to teach it a consistent response pattern for interpreting structured metrics.

Each training example includes a system message, a user message containing experiment metrics, and an assistant response with a structured recommendation. The examples include different result patterns, such as statistically significant wins, non-significant results, negative lift, and cases where more testing would be needed.

The expected output format is intentionally consistent so the fine-tuned model can learn the preferred structure.

In [2]:
training_examples = [
    {
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a digital experimentation analyst. "
                    "Interpret A/B test results using the provided metrics."
                )
            },
            {
                "role": "user",
                "content": (
                    "Control Conversion Rate: 10.2%\n"
                    "Variant Conversion Rate: 11.8%\n"
                    "Sample Size: 8,000 per group\n"
                    "P-value: 0.04\n"
                    "Lift: +15.7%"
                )
            },
            {
                "role": "assistant",
                "content": (
                    "Summary: The variant outperformed the control with a conversion "
                    "rate of 11.8% compared to 10.2%.\n"
                    "Interpretation: The result is statistically significant because "
                    "the p-value is below 0.05, and the 15.7% lift suggests a "
                    "meaningful improvement.\n"
                    "Recommendation: Implement the variant while continuing to "
                    "monitor performance after rollout."
                )
            }
        ]
    },

    {
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a digital experimentation analyst. "
                    "Interpret A/B test results using the provided metrics."
                )
            },
            {
                "role": "user",
                "content": (
                    "Control Conversion Rate: 12.4%\n"
                    "Variant Conversion Rate: 12.7%\n"
                    "Sample Size: 5,000 per group\n"
                    "P-value: 0.42\n"
                    "Lift: +2.4%"
                )
            },
            {
                "role": "assistant",
                "content": (
                    "Summary: The variant performed slightly better than the control, "
                    "but the difference was small.\n"
                    "Interpretation: The result is not statistically significant "
                    "because the p-value is above 0.05, and the lift is modest.\n"
                    "Recommendation: Do not implement the variant. Consider "
                    "additional testing or a stronger design change."
                )
            }
        ]
    },

    {
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a digital experimentation analyst. "
                    "Interpret A/B test results using the provided metrics."
                )
            },
            {
                "role": "user",
                "content": (
                    "Control Conversion Rate: 9.5%\n"
                    "Variant Conversion Rate: 8.9%\n"
                    "Sample Size: 7,500 per group\n"
                    "P-value: 0.03\n"
                    "Lift: -6.3%"
                )
            },
            {
                "role": "assistant",
                "content": (
                    "Summary: The variant underperformed the control with a lower "
                    "conversion rate.\n"
                    "Interpretation: The negative lift is statistically significant "
                    "because the p-value is below 0.05.\n"
                    "Recommendation: Do not implement the variant. Review the "
                    "experience to understand why performance declined."
                )
            }
        ]
    },

    {
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a digital experimentation analyst. "
                    "Interpret A/B test results using the provided metrics."
                )
            },
            {
                "role": "user",
                "content": (
                    "Control Conversion Rate: 15.0%\n"
                    "Variant Conversion Rate: 16.1%\n"
                    "Sample Size: 2,000 per group\n"
                    "P-value: 0.08\n"
                    "Lift: +7.3%"
                )
            },
            {
                "role": "assistant",
                "content": (
                    "Summary: The variant showed a positive lift compared to the "
                    "control.\n"
                    "Interpretation: The improvement may be meaningful, but the "
                    "result is not statistically significant because the p-value is "
                    "above 0.05.\n"
                    "Recommendation: Do not implement yet. Continue testing or "
                    "collect more data before making a decision."
                )
            }
        ]
    },

    {
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a digital experimentation analyst. "
                    "Interpret A/B test results using the provided metrics."
                )
            },
            {
                "role": "user",
                "content": (
                    "Control Conversion Rate: 6.8%\n"
                    "Variant Conversion Rate: 7.5%\n"
                    "Sample Size: 10,000 per group\n"
                    "P-value: 0.01\n"
                    "Lift: +10.3%"
                )
            },
            {
                "role": "assistant",
                "content": (
                    "Summary: The variant produced a higher conversion rate than the "
                    "control.\n"
                    "Interpretation: The result is statistically significant, and "
                    "the lift indicates a meaningful improvement.\n"
                    "Recommendation: Implement the variant and monitor "
                    "post-launch performance."
                )
            }
        ]
    }
]

print("Training examples created:", len(training_examples))
print(training_examples[0])

Training examples created: 5
{'messages': [{'role': 'system', 'content': 'You are a digital experimentation analyst. Interpret A/B test results using the provided metrics.'}, {'role': 'user', 'content': 'Control Conversion Rate: 10.2%\nVariant Conversion Rate: 11.8%\nSample Size: 8,000 per group\nP-value: 0.04\nLift: +15.7%'}, {'role': 'assistant', 'content': 'Summary: The variant outperformed the control with a conversion rate of 11.8% compared to 10.2%.\nInterpretation: The result is statistically significant because the p-value is below 0.05, and the 15.7% lift suggests a meaningful improvement.\nRecommendation: Implement the variant while continuing to monitor performance after rollout.'}]}


### Dataset Creation Result

A total of five training examples were successfully created. Each example follows the expected message format, including a system role to define behavior, a user input containing A/B test metrics, and an assistant response with a structured interpretation.

The dataset includes a mix of scenarios, such as statistically significant improvements, non-significant results, negative lift, and cases where additional testing is recommended. This variation is intentional so the model can learn how to respond across different types of outcomes rather than only successful tests.

The assistant responses follow a consistent structure using Summary, Interpretation, and Recommendation sections. This consistency is important because it reinforces the desired output format during fine-tuning.

While the initial dataset is relatively small, it provides a starting point for testing the fine-tuning workflow. Later validation determines whether additional examples are required.

## Validation Dataset Design

A separate validation dataset is created to test the fine-tuned model. These examples are not included in the training data, which allows for a more realistic evaluation of how the model performs on new inputs.

The validation examples follow the same structure as the training data but use different values to represent new scenarios. This helps assess whether the model generalizes the learned patterns rather than memorizing specific cases.

In [3]:
validation_examples = [
    {
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a digital experimentation analyst. "
                    "Interpret A/B test results using the provided metrics."
                )
            },
            {
                "role": "user",
                "content": (
                    "Control Conversion Rate: 11.0%\n"
                    "Variant Conversion Rate: 12.2%\n"
                    "Sample Size: 6,000 per group\n"
                    "P-value: 0.02\n"
                    "Lift: +10.9%"
                )
            }
        ]
    },
    {
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a digital experimentation analyst. "
                    "Interpret A/B test results using the provided metrics."
                )
            },
            {
                "role": "user",
                "content": (
                    "Control Conversion Rate: 14.5%\n"
                    "Variant Conversion Rate: 14.8%\n"
                    "Sample Size: 4,000 per group\n"
                    "P-value: 0.35\n"
                    "Lift: +2.1%"
                )
            }
        ]
    },
    {
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a digital experimentation analyst. "
                    "Interpret A/B test results using the provided metrics."
                )
            },
            {
                "role": "user",
                "content": (
                    "Control Conversion Rate: 8.2%\n"
                    "Variant Conversion Rate: 7.6%\n"
                    "Sample Size: 9,000 per group\n"
                    "P-value: 0.01\n"
                    "Lift: -7.3%"
                )
            }
        ]
    }
]

print("Validation examples created:", len(validation_examples))
print(validation_examples[0])

Validation examples created: 3
{'messages': [{'role': 'system', 'content': 'You are a digital experimentation analyst. Interpret A/B test results using the provided metrics.'}, {'role': 'user', 'content': 'Control Conversion Rate: 11.0%\nVariant Conversion Rate: 12.2%\nSample Size: 6,000 per group\nP-value: 0.02\nLift: +10.9%'}]}


### Validation Dataset Result

A total of three validation examples were created. These examples were initially created without assistant responses as an attempt to simulate test prompts. However, this approach was later corrected after the fine-tuning job returned a validation format error requiring assistant responses.

The scenarios include a statistically significant positive result, a non-significant result, and a statistically significant negative outcome. This variation allows the evaluation to cover multiple decision paths, including implementation, rejection, and the need for further testing.

By separating validation data from training data, the evaluation can better reflect how the fine-tuned model performs on unseen inputs. This helps determine whether the model has learned general decision patterns rather than memorizing specific examples.

## Save Training and Validation Files

The training and validation datasets are saved as JSONL files, which is the required format for OpenAI fine-tuning. Each line in the file represents a single training example.

These files will be uploaded to OpenAI in the next step.

In [4]:
def save_jsonl(data, filename):
    path = Path(filename)
    with path.open("w", encoding="utf-8") as f:
        for entry in data:
            f.write(json.dumps(entry) + "\n")
    return path


train_file = save_jsonl(
    training_examples,
    "training_data.jsonl"
)

val_file = save_jsonl(
    validation_examples,
    "validation_data.jsonl"
)

print("Training file:", train_file)
print("Validation file:", val_file)

Training file: training_data.jsonl
Validation file: validation_data.jsonl


### File Creation Result

The training and validation datasets were successfully saved as JSONL files. Each file contains one example per line, which matches the JSONL format required for OpenAI fine-tuning. However, while the file structure was correct, the validation file later required updates to include assistant responses to meet fine-tuning requirements.

Separating the data into two files allows the training process to learn from one dataset while reserving another for evaluation. This setup helps ensure that model performance can be assessed on unseen examples rather than the same data used during training.

## Upload Files to OpenAI

The training and validation JSONL files are uploaded to OpenAI so they can be used in the fine-tuning process.

Each file is uploaded with the purpose set to "fine-tune", which allows it to be used when creating the fine-tuning job.

In [5]:
train_upload = client.files.create(
    file=open("training_data.jsonl", "rb"),
    purpose="fine-tune"
)

val_upload = client.files.create(
    file=open("validation_data.jsonl", "rb"),
    purpose="fine-tune"
)

print("Training File ID:", train_upload.id)
print("Validation File ID:", val_upload.id)

Training File ID: file-WQ9T5HxQLpRo9GCvH6gZhd
Validation File ID: file-2rtVENvGVjWfnC7WqPW3dg


### Upload Result

Both JSONL files were successfully uploaded to OpenAI. The returned file IDs confirm that the training and validation files were successfully uploaded. However, successful upload does not guarantee that the files meet all fine-tuning requirements, which was later validated during job execution.

This step is important because it verifies that the files were formatted correctly enough for OpenAI to accept them for fine-tuning.

## Create Fine-Tuning Job

This step creates the fine-tuning job using the uploaded training and validation files. The training file is used to teach the model the desired response pattern, while the validation file is used to evaluate performance on separate examples.

The goal is to create a model that consistently returns structured A/B test interpretations using the Summary, Interpretation, and Recommendation format.

In [6]:
fine_tune_job = client.fine_tuning.jobs.create(
    training_file=train_upload.id,
    validation_file=val_upload.id,
    model="gpt-4o-mini-2024-07-18",
    suffix="experiment-analysis"
)

print("Fine-tuning job created.")
print("Job ID:", fine_tune_job.id)
print("Status:", fine_tune_job.status)

Fine-tuning job created.
Job ID: ftjob-HiO6XnjIX2insdl8aCeAiC8s
Status: validating_files


### Fine-Tuning Job Creation Result

The fine-tuning job was successfully created. The initial status was `validating_files`, which means OpenAI accepted the request and began checking the uploaded training and validation files before starting the training process.

At this stage, the model has not started training yet. The system is first verifying that the files are usable for fine-tuning. 

At this point, the job had not yet validated the internal structure of the files, which led to the next observed failure.

## Track Fine-Tuning Job Status

After creating the fine-tuning job, the next step is to check the job status. This helps document the model build process and shows whether the job progresses from file validation into training and completion.

Tracking the job status is also useful because fine-tuning is not always immediate. Depending on system availability and file validation, the job may move through several states before finishing.

In [7]:
job_status = client.fine_tuning.jobs.retrieve(
    fine_tune_job.id
)

print("Job ID:", job_status.id)
print("Status:", job_status.status)
print("Base Model:", job_status.model)
print("Fine-Tuned Model:", job_status.fine_tuned_model)

Job ID: ftjob-HiO6XnjIX2insdl8aCeAiC8s
Status: failed
Base Model: gpt-4o-mini-2024-07-18
Fine-Tuned Model: None


In [8]:
job_status = client.fine_tuning.jobs.retrieve(
    fine_tune_job.id
)

print("Status:", job_status.status)
print("Error:", job_status.error)

Status: failed
Error: Error(code='invalid_file_format', message='The job failed due to a file format error in the validation file. Invalid file format. Example 1, The last message must be from the assistant.', param='validation_file')


### First Fine-Tuning Attempt Result

The first fine-tuning job failed during file validation. The error message indicated that the validation file was not in the expected format because the last message in each validation example must be from the assistant.

This was a useful checkpoint. I had originally structured the validation examples like test prompts, but OpenAI's fine-tuning validation file expects complete examples with both the user input and the expected assistant response. The validation file is used during training evaluation, not as an open-ended test set.

In [9]:
validation_examples = [
    {
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a digital experimentation analyst. "
                    "Interpret A/B test results using the provided metrics."
                )
            },
            {
                "role": "user",
                "content": (
                    "Control Conversion Rate: 11.0%\n"
                    "Variant Conversion Rate: 12.2%\n"
                    "Sample Size: 6,000 per group\n"
                    "P-value: 0.02\n"
                    "Lift: +10.9%"
                )
            },
            {
                "role": "assistant",
                "content": (
                    "Summary: The variant outperformed the control with a higher "
                    "conversion rate.\n"
                    "Interpretation: The result is statistically significant because "
                    "the p-value is below 0.05, and the positive lift suggests a "
                    "meaningful improvement.\n"
                    "Recommendation: Implement the variant and monitor performance "
                    "after rollout."
                )
            }
        ]
    },
    {
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a digital experimentation analyst. "
                    "Interpret A/B test results using the provided metrics."
                )
            },
            {
                "role": "user",
                "content": (
                    "Control Conversion Rate: 14.5%\n"
                    "Variant Conversion Rate: 14.8%\n"
                    "Sample Size: 4,000 per group\n"
                    "P-value: 0.35\n"
                    "Lift: +2.1%"
                )
            },
            {
                "role": "assistant",
                "content": (
                    "Summary: The variant performed slightly better than the control, "
                    "but the difference was small.\n"
                    "Interpretation: The result is not statistically significant "
                    "because the p-value is above 0.05, and the lift is modest.\n"
                    "Recommendation: Do not implement the variant based on this test. "
                    "Consider additional testing or a stronger change."
                )
            }
        ]
    },
    {
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a digital experimentation analyst. "
                    "Interpret A/B test results using the provided metrics."
                )
            },
            {
                "role": "user",
                "content": (
                    "Control Conversion Rate: 8.2%\n"
                    "Variant Conversion Rate: 7.6%\n"
                    "Sample Size: 9,000 per group\n"
                    "P-value: 0.01\n"
                    "Lift: -7.3%"
                )
            },
            {
                "role": "assistant",
                "content": (
                    "Summary: The variant underperformed the control.\n"
                    "Interpretation: The negative lift is statistically significant "
                    "because the p-value is below 0.05.\n"
                    "Recommendation: Do not implement the variant. Investigate the "
                    "experience to understand why performance declined."
                )
            }
        ]
    }
]

print("Corrected validation examples created:", len(validation_examples))
print(validation_examples[0])

Corrected validation examples created: 3
{'messages': [{'role': 'system', 'content': 'You are a digital experimentation analyst. Interpret A/B test results using the provided metrics.'}, {'role': 'user', 'content': 'Control Conversion Rate: 11.0%\nVariant Conversion Rate: 12.2%\nSample Size: 6,000 per group\nP-value: 0.02\nLift: +10.9%'}, {'role': 'assistant', 'content': 'Summary: The variant outperformed the control with a higher conversion rate.\nInterpretation: The result is statistically significant because the p-value is below 0.05, and the positive lift suggests a meaningful improvement.\nRecommendation: Implement the variant and monitor performance after rollout.'}]}


### Corrected Validation Dataset Result

The validation dataset was updated to include expected assistant responses. This corrected the file structure so that each validation example now includes the same three-part message pattern used in the training data: system instruction, user input, and assistant response.

This correction clarified the difference between a validation file for fine-tuning and a test prompt used after training. The validation file needs complete examples so OpenAI can evaluate model behavior during the fine-tuning process.

In [10]:
train_file = save_jsonl(
    training_examples,
    "training_data.jsonl"
)

val_file = save_jsonl(
    validation_examples,
    "validation_data.jsonl"
)

print("Training file:", train_file)
print("Corrected validation file:", val_file)

Training file: training_data.jsonl
Corrected validation file: validation_data.jsonl


In [11]:
val_upload = client.files.create(
    file=open("validation_data.jsonl", "rb"),
    purpose="fine-tune"
)

print("Corrected Validation File ID:", val_upload.id)

Corrected Validation File ID: file-1P1JkRNefdT9QRjjaYweCS


### Validation File Correction Result

The corrected validation dataset was successfully saved and re-uploaded to OpenAI. A new file ID was generated, confirming that the updated file is now available for fine-tuning.

This step resolved the earlier formatting issue and ensures that both the training and validation files meet the required structure for the fine-tuning process.

## Retry Fine-Tuning Job

After correcting the validation dataset format, the fine-tuning job is created again using the updated validation file.

This step verifies that the corrected data structure resolves the previous error and allows the training process to proceed.

In [12]:
fine_tune_job = client.fine_tuning.jobs.create(
    training_file=train_upload.id,
    validation_file=val_upload.id,
    model="gpt-4o-mini-2024-07-18",
    suffix="experiment-analysis-v2"
)

print("Fine-tuning job created.")
print("Job ID:", fine_tune_job.id)
print("Status:", fine_tune_job.status)

Fine-tuning job created.
Job ID: ftjob-Z4j11r7kGrfNP3GmMPfrS3HP
Status: validating_files


### Retry Result

The second fine-tuning job was created successfully and entered the `validating_files` status. This indicates that the corrected validation file was accepted far enough for OpenAI to begin the validation process again.

This retry confirms that the earlier issue was related to the validation file structure rather than the overall fine-tuning setup.

In [13]:
job_status = client.fine_tuning.jobs.retrieve(
    fine_tune_job.id
)

print("Job ID:", job_status.id)
print("Status:", job_status.status)
print("Base Model:", job_status.model)
print("Fine-Tuned Model:", job_status.fine_tuned_model)
print("Error:", job_status.error)

Job ID: ftjob-Z4j11r7kGrfNP3GmMPfrS3HP
Status: failed
Base Model: gpt-4o-mini-2024-07-18
Fine-Tuned Model: None
Error: Error(code='invalid_n_examples', message='Training file has 5 example(s), but must have at least 10 examples', param='training_file')


### Second Fine-Tuning Attempt Result

The second fine-tuning job passed the earlier validation-file issue but failed because the training file only contained five examples. The error message indicated that the training file must contain at least ten examples.

This result clarified a minimum dataset-size requirement for fine-tuning. The dataset structure is now correct, but more training examples are needed before the job can run successfully.

In [14]:
training_examples = [
    {
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a digital experimentation analyst. "
                    "Interpret A/B test results using the provided metrics."
                )
            },
            {
                "role": "user",
                "content": (
                    "Control Conversion Rate: 10.2%\n"
                    "Variant Conversion Rate: 11.8%\n"
                    "Sample Size: 8,000 per group\n"
                    "P-value: 0.04\n"
                    "Lift: +15.7%"
                )
            },
            {
                "role": "assistant",
                "content": (
                    "Summary: The variant outperformed the control with a higher "
                    "conversion rate.\n"
                    "Interpretation: The result is statistically significant because "
                    "the p-value is below 0.05, and the positive lift suggests a "
                    "meaningful improvement.\n"
                    "Recommendation: Implement the variant while monitoring "
                    "post-launch performance."
                )
            }
        ]
    },
    {
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a digital experimentation analyst. "
                    "Interpret A/B test results using the provided metrics."
                )
            },
            {
                "role": "user",
                "content": (
                    "Control Conversion Rate: 12.4%\n"
                    "Variant Conversion Rate: 12.7%\n"
                    "Sample Size: 5,000 per group\n"
                    "P-value: 0.42\n"
                    "Lift: +2.4%"
                )
            },
            {
                "role": "assistant",
                "content": (
                    "Summary: The variant performed slightly better than the control, "
                    "but the difference was small.\n"
                    "Interpretation: The result is not statistically significant "
                    "because the p-value is above 0.05, and the lift is modest.\n"
                    "Recommendation: Do not implement the variant based on this test. "
                    "Consider additional testing or a stronger design change."
                )
            }
        ]
    },
    {
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a digital experimentation analyst. "
                    "Interpret A/B test results using the provided metrics."
                )
            },
            {
                "role": "user",
                "content": (
                    "Control Conversion Rate: 9.5%\n"
                    "Variant Conversion Rate: 8.9%\n"
                    "Sample Size: 7,500 per group\n"
                    "P-value: 0.03\n"
                    "Lift: -6.3%"
                )
            },
            {
                "role": "assistant",
                "content": (
                    "Summary: The variant underperformed the control with a lower "
                    "conversion rate.\n"
                    "Interpretation: The negative lift is statistically significant "
                    "because the p-value is below 0.05.\n"
                    "Recommendation: Do not implement the variant. Review the "
                    "experience to understand why performance declined."
                )
            }
        ]
    },
    {
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a digital experimentation analyst. "
                    "Interpret A/B test results using the provided metrics."
                )
            },
            {
                "role": "user",
                "content": (
                    "Control Conversion Rate: 15.0%\n"
                    "Variant Conversion Rate: 16.1%\n"
                    "Sample Size: 2,000 per group\n"
                    "P-value: 0.08\n"
                    "Lift: +7.3%"
                )
            },
            {
                "role": "assistant",
                "content": (
                    "Summary: The variant showed a positive lift compared to the "
                    "control.\n"
                    "Interpretation: The improvement may be meaningful, but the "
                    "result is not statistically significant because the p-value is "
                    "above 0.05.\n"
                    "Recommendation: Do not implement yet. Continue testing or "
                    "collect more data before making a decision."
                )
            }
        ]
    },
    {
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a digital experimentation analyst. "
                    "Interpret A/B test results using the provided metrics."
                )
            },
            {
                "role": "user",
                "content": (
                    "Control Conversion Rate: 6.8%\n"
                    "Variant Conversion Rate: 7.5%\n"
                    "Sample Size: 10,000 per group\n"
                    "P-value: 0.01\n"
                    "Lift: +10.3%"
                )
            },
            {
                "role": "assistant",
                "content": (
                    "Summary: The variant produced a higher conversion rate than the "
                    "control.\n"
                    "Interpretation: The result is statistically significant, and "
                    "the lift indicates a meaningful improvement.\n"
                    "Recommendation: Implement the variant and monitor "
                    "post-launch performance."
                )
            }
        ]
    },
    {
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a digital experimentation analyst. "
                    "Interpret A/B test results using the provided metrics."
                )
            },
            {
                "role": "user",
                "content": (
                    "Control Conversion Rate: 18.2%\n"
                    "Variant Conversion Rate: 17.9%\n"
                    "Sample Size: 6,500 per group\n"
                    "P-value: 0.51\n"
                    "Lift: -1.6%"
                )
            },
            {
                "role": "assistant",
                "content": (
                    "Summary: The variant performed slightly worse than the control.\n"
                    "Interpretation: The result is not statistically significant "
                    "because the p-value is above 0.05, and the negative lift is "
                    "small.\n"
                    "Recommendation: Do not implement the variant. Treat the result "
                    "as inconclusive and review whether the change is worth retesting."
                )
            }
        ]
    },
    {
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a digital experimentation analyst. "
                    "Interpret A/B test results using the provided metrics."
                )
            },
            {
                "role": "user",
                "content": (
                    "Control Conversion Rate: 4.1%\n"
                    "Variant Conversion Rate: 4.9%\n"
                    "Sample Size: 12,000 per group\n"
                    "P-value: 0.02\n"
                    "Lift: +19.5%"
                )
            },
            {
                "role": "assistant",
                "content": (
                    "Summary: The variant produced a higher conversion rate than the "
                    "control.\n"
                    "Interpretation: The result is statistically significant because "
                    "the p-value is below 0.05, and the lift is large enough to be "
                    "practically meaningful.\n"
                    "Recommendation: Implement the variant and continue monitoring "
                    "performance after launch."
                )
            }
        ]
    },
    {
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a digital experimentation analyst. "
                    "Interpret A/B test results using the provided metrics."
                )
            },
            {
                "role": "user",
                "content": (
                    "Control Conversion Rate: 22.0%\n"
                    "Variant Conversion Rate: 22.4%\n"
                    "Sample Size: 15,000 per group\n"
                    "P-value: 0.06\n"
                    "Lift: +1.8%"
                )
            },
            {
                "role": "assistant",
                "content": (
                    "Summary: The variant performed slightly better than the control.\n"
                    "Interpretation: The result is not statistically significant "
                    "because the p-value is above 0.05, and the lift is small.\n"
                    "Recommendation: Do not implement the variant based on this "
                    "result. Consider whether the change should be revised or tested "
                    "again with a clearer expected impact."
                )
            }
        ]
    },
    {
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a digital experimentation analyst. "
                    "Interpret A/B test results using the provided metrics."
                )
            },
            {
                "role": "user",
                "content": (
                    "Control Conversion Rate: 7.3%\n"
                    "Variant Conversion Rate: 6.4%\n"
                    "Sample Size: 5,500 per group\n"
                    "P-value: 0.04\n"
                    "Lift: -12.3%"
                )
            },
            {
                "role": "assistant",
                "content": (
                    "Summary: The variant produced a lower conversion rate than the "
                    "control.\n"
                    "Interpretation: The decline is statistically significant because "
                    "the p-value is below 0.05, and the negative lift suggests a "
                    "meaningful performance drop.\n"
                    "Recommendation: Do not implement the variant. Investigate the "
                    "change before considering any further testing."
                )
            }
        ]
    },
    {
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a digital experimentation analyst. "
                    "Interpret A/B test results using the provided metrics."
                )
            },
            {
                "role": "user",
                "content": (
                    "Control Conversion Rate: 13.6%\n"
                    "Variant Conversion Rate: 14.9%\n"
                    "Sample Size: 3,000 per group\n"
                    "P-value: 0.04\n"
                    "Lift: +9.6%"
                )
            },
            {
                "role": "assistant",
                "content": (
                    "Summary: The variant outperformed the control with a higher "
                    "conversion rate.\n"
                    "Interpretation: The result is statistically significant because "
                    "the p-value is below 0.05, and the lift suggests a meaningful "
                    "improvement.\n"
                    "Recommendation: Implement the variant, but monitor performance "
                    "because the sample size is smaller than other examples."
                )
            }
        ]
    },
    {
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a digital experimentation analyst. "
                    "Interpret A/B test results using the provided metrics."
                )
            },
            {
                "role": "user",
                "content": (
                    "Control Conversion Rate: 5.5%\n"
                    "Variant Conversion Rate: 5.7%\n"
                    "Sample Size: 9,500 per group\n"
                    "P-value: 0.28\n"
                    "Lift: +3.6%"
                )
            },
            {
                "role": "assistant",
                "content": (
                    "Summary: The variant showed a small improvement over the control.\n"
                    "Interpretation: The result is not statistically significant "
                    "because the p-value is above 0.05, and the lift is modest.\n"
                    "Recommendation: Do not implement the variant based on this test. "
                    "Consider a revised hypothesis or further experimentation."
                )
            }
        ]
    },
    {
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a digital experimentation analyst. "
                    "Interpret A/B test results using the provided metrics."
                )
            },
            {
                "role": "user",
                "content": (
                    "Control Conversion Rate: 16.8%\n"
                    "Variant Conversion Rate: 18.4%\n"
                    "Sample Size: 20,000 per group\n"
                    "P-value: 0.001\n"
                    "Lift: +9.5%"
                )
            },
            {
                "role": "assistant",
                "content": (
                    "Summary: The variant clearly outperformed the control.\n"
                    "Interpretation: The result is statistically significant because "
                    "the p-value is well below 0.05, and the lift indicates a "
                    "meaningful improvement.\n"
                    "Recommendation: Implement the variant and monitor post-launch "
                    "performance."
                )
            }
        ]
    }
]

print("Expanded training examples created:", len(training_examples))

Expanded training examples created: 12


### Expanded Training Dataset Result

The training dataset was expanded from five examples to twelve examples. This resolved the minimum training example issue identified during the second fine-tuning attempt.

The expanded dataset includes a wider range of experiment outcomes, including statistically significant wins, non-significant positive lifts, significant negative results, and small inconclusive changes. This broader range should help the model learn the intended decision logic more effectively.

Even though the dataset is still small, it is appropriate for this milestone because the goal is to fine-tune for response consistency and structure rather than to build a broad statistical reasoning model.

In [15]:
train_file = save_jsonl(
    training_examples,
    "training_data.jsonl"
)

val_file = save_jsonl(
    validation_examples,
    "validation_data.jsonl"
)

print("Expanded training file:", train_file)
print("Validation file:", val_file)

Expanded training file: training_data.jsonl
Validation file: validation_data.jsonl


In [16]:
train_upload = client.files.create(
    file=open("training_data.jsonl", "rb"),
    purpose="fine-tune"
)

print("Expanded Training File ID:", train_upload.id)

Expanded Training File ID: file-YYaAoPKjawuEaTAScyb7Ak


In [17]:
fine_tune_job = client.fine_tuning.jobs.create(
    training_file=train_upload.id,
    validation_file=val_upload.id,
    model="gpt-4o-mini-2024-07-18",
    suffix="experiment-analysis-v3"
)

print("Fine-tuning job created.")
print("Job ID:", fine_tune_job.id)
print("Status:", fine_tune_job.status)

Fine-tuning job created.
Job ID: ftjob-tJLwoOjz0NQ2VcOwnhsEIFNd
Status: validating_files


### Final Fine-Tuning Job Creation Result

The training dataset was successfully expanded and re-uploaded, and a new fine-tuning job was created using the updated training file and corrected validation file.

The job entered the `validating_files` status, indicating that both files were accepted and the system has begun preparing for training. This confirms that the earlier issues related to dataset size and validation format have been resolved.

At this point, the model is expected to proceed through validation and into the training phase.

## Monitor Fine-Tuning Progress

The fine-tuning job progresses through several stages before completion. These include file validation, training, and final model creation.

The job status is checked periodically until the process completes. Once finished, the fine-tuned model ID will be available for testing.

In [29]:
job_status = client.fine_tuning.jobs.retrieve(
    fine_tune_job.id
)

print("Status:", job_status.status)
print("Fine-Tuned Model:", job_status.fine_tuned_model)
print("Error:", job_status.error)

Status: succeeded
Fine-Tuned Model: ft:gpt-4o-mini-2024-07-18:personal:experiment-analysis-v3:DbSPQHKQ
Error: Error(code=None, message=None, param=None)


### Fine-Tuning Completion Result

The fine-tuning job completed successfully. The output includes a fine-tuned model ID, which confirms that OpenAI created a custom model from the training data.

This is the first completed model build for the project. The next step is to review the job events and available metrics, then test the fine-tuned model against a baseline prompt-based model.

In [30]:
fine_tuned_model = job_status.fine_tuned_model

print("Fine-tuned model ID:")
print(fine_tuned_model)

Fine-tuned model ID:
ft:gpt-4o-mini-2024-07-18:personal:experiment-analysis-v3:DbSPQHKQ


## Review Fine-Tuning Events

The fine-tuning events provide a record of the model build process. Reviewing these events helps document the major steps that occurred during validation, training, and completion.

In [31]:
events = client.fine_tuning.jobs.list_events(
    fine_tuning_job_id=fine_tune_job.id,
    limit=10
)

for event in events.data:
    print(event.created_at, "-", event.message)

1777820440 - The job has successfully completed
1777820436 - Usage policy evaluations completed, model is now enabled for sampling
1777820436 - Moderation checks for snapshot ft:gpt-4o-mini-2024-07-18:personal:experiment-analysis-v3:DbSPQHKQ passed.
1777819712 - Evaluating model against our usage policies
1777819712 - New fine-tuned model created
1777819712 - Checkpoint created at step 84
1777819712 - Checkpoint created at step 72
1777819703 - Step 96/96: training loss=0.03, validation loss=0.27, full validation loss=0.24
1777819700 - Step 95/96: training loss=0.06, validation loss=0.10
1777819700 - Step 94/96: training loss=0.07, validation loss=0.26


### Fine-Tuning Events and Metrics

The fine-tuning event log confirms that the job completed successfully and that a new fine-tuned model was created. The events also show that checkpoints were created during training and that the model passed moderation and usage policy checks before being enabled for sampling.

The final training step reported a training loss of 0.03, validation loss of 0.27, and full validation loss of 0.24. The Chapter 9 fine-tuning metrics discussion identifies training loss, validation loss, mean token accuracy, and token accuracy as key metrics. In this fine-tuning run, the event log returned training loss, validation loss, and full validation loss, but it did not display mean token accuracy or token accuracy in the visible output. Because of that, this notebook documents and interprets the metrics that were returned by the OpenAI fine-tuning event endpoint.

The low training loss suggests that the model learned the patterns in the training examples well. The validation loss is higher than the training loss, which is expected with a small dataset and indicates that performance on unseen examples is more limited than performance on the training examples.

Overall, the metrics suggest that the fine-tuning process worked, but the difference between training and validation loss reinforces that this should be treated as a small experimental fine-tune rather than a production-ready model.

## Baseline and Fine-Tuned Model Comparison

To evaluate the fine-tuned model, the same unseen A/B test input is sent to both the baseline model and the fine-tuned model. This allows the outputs to be compared directly.

The evaluation focuses on structure, recommendation quality, consistency, and whether the output follows the intended Summary, Interpretation, and Recommendation format.

In [32]:
test_input = (
    "Control Conversion Rate: 9.8%\n"
    "Variant Conversion Rate: 10.9%\n"
    "Sample Size: 7,000 per group\n"
    "P-value: 0.03\n"
    "Lift: +11.2%"
)

comparison_prompt = (
    "Interpret the following A/B test results and provide a recommendation.\n\n"
    f"{test_input}"
)


def get_model_response(model_name, prompt):
    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a digital experimentation analyst. "
                    "Interpret A/B test results using the provided metrics."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    return response.choices[0].message.content


baseline_output = get_model_response(
    "gpt-4o-mini",
    comparison_prompt
)

fine_tuned_output = get_model_response(
    fine_tuned_model,
    comparison_prompt
)

print("BASELINE MODEL OUTPUT")
print("---------------------")
print(baseline_output)

print("\nFINE-TUNED MODEL OUTPUT")
print("-----------------------")
print(fine_tuned_output)

BASELINE MODEL OUTPUT
---------------------
Based on the A/B test results provided, we can interpret the findings as follows:

1. **Control Conversion Rate**: The control group had a conversion rate of 9.8%.
2. **Variant Conversion Rate**: The variant group had a conversion rate of 10.9%.
3. **Sample Size**: Each group (control and variant) had a sample size of 7,000, which is a substantial number that can provide reliable results.
4. **P-value**: The p-value of 0.03 indicates that there is a statistically significant difference between the control and variant groups. Typically, a p-value less than 0.05 is considered significant, suggesting that the observed difference in conversion rates is unlikely to be due to random chance.
5. **Lift**: The variant shows a lift of +11.2% compared to the control, indicating that the variant performed better in terms of conversions.

### Recommendation:
Given the statistically significant increase in conversion rate (p-value of 0.03) and the positive

### Comparison Result

Both models reached the correct recommendation. The baseline model correctly identified the statistically significant result and recommended implementing the variant. However, the baseline response was much longer and introduced extra explanation, including potential business impacts such as revenue and customer engagement that were not provided in the input data.

The fine-tuned model produced a much shorter and more consistent response. It followed the intended Summary, Interpretation, and Recommendation structure exactly. It also avoided adding unnecessary assumptions beyond the provided metrics.

This comparison suggests that fine-tuning improved output consistency and formatting more than decision accuracy. The baseline model was already capable of interpreting the result correctly, but the fine-tuned model better matched the desired communication style for this project.

## Second Test Case: Positive Lift but Not Statistically Significant

The second test case evaluates how both models handle a more cautious scenario. The variant performs slightly better than the control, but the p-value is above 0.05.

This case is important because an accurate recommendation should avoid implementation even though the lift is positive.

In [33]:
test_input_2 = (
    "Control Conversion Rate: 13.5%\n"
    "Variant Conversion Rate: 13.9%\n"
    "Sample Size: 5,500 per group\n"
    "P-value: 0.31\n"
    "Lift: +3.0%"
)

comparison_prompt_2 = (
    "Interpret the following A/B test results and provide a recommendation.\n\n"
    f"{test_input_2}"
)

baseline_output_2 = get_model_response(
    "gpt-4o-mini",
    comparison_prompt_2
)

fine_tuned_output_2 = get_model_response(
    fine_tuned_model,
    comparison_prompt_2
)

print("BASELINE MODEL OUTPUT")
print("---------------------")
print(baseline_output_2)

print("\nFINE-TUNED MODEL OUTPUT")
print("-----------------------")
print(fine_tuned_output_2)

BASELINE MODEL OUTPUT
---------------------
Based on the A/B test results provided, we can analyze the performance of the control group versus the variant group.

1. **Conversion Rates**:
   - Control Group: 13.5%
   - Variant Group: 13.9%
   - The variant group shows a conversion rate that is 0.4 percentage points higher than the control group, which translates to a lift of +3.0%.

2. **Sample Size**:
   - Both groups had a sample size of 5,500, which is a reasonable size for detecting differences in conversion rates.

3. **P-value**:
   - The p-value is 0.31, which is significantly higher than the common alpha level of 0.05. This indicates that the difference in conversion rates between the control and variant groups is not statistically significant. In other words, there is a 31% probability that the observed difference could be due to random chance rather than a true effect of the variant.

### Interpretation:
- While the variant group shows a slight improvement in conversion rate 

### Second Test Case Result

Both models correctly handled the more cautious scenario. The baseline model identified that the positive lift was not statistically significant and appropriately recommended not implementing the variant. It also provided additional suggestions such as increasing sample size and exploring alternative changes.

The fine-tuned model again produced a shorter and more structured response. It clearly followed the Summary, Interpretation, and Recommendation format and reached the same decision without adding extra explanation beyond the provided metrics.

This result reinforces the earlier finding that fine-tuning does not significantly improve decision accuracy. The baseline model is already capable of making correct recommendations in both strong and ambiguous scenarios. However, the fine-tuned model consistently produces more controlled and predictable outputs, which is valuable for applications that require standardized responses.

## Overall Evaluation

The results of the fine-tuning process show that the model was successfully trained to produce more consistent and structured outputs. The fine-tuned model reliably followed the intended Summary, Interpretation, and Recommendation format across all test cases, demonstrating that it learned the desired response pattern.

However, the comparison tests indicate that fine-tuning did not meaningfully improve decision accuracy. The baseline model was already capable of correctly interpreting A/B test results and making appropriate recommendations in both statistically significant and non-significant scenarios. In both test cases, the baseline and fine-tuned models reached the same conclusions.

The primary benefit of fine-tuning in this project is improved output control rather than improved reasoning. The fine-tuned model produces shorter, more predictable responses and avoids adding assumptions beyond the provided data. This makes the output easier to standardize and more suitable for integration into structured workflows or applications.

The training and validation metrics also highlight the limitations of the approach. While the model achieved a low training loss, the higher validation loss suggests reduced performance on unseen examples. This gap reflects the small size of the training dataset and indicates that the model has limited generalization ability. As a result, this fine-tuned model should be viewed as an experimental implementation rather than a production-ready solution.

Overall, prompt engineering alone was sufficient to achieve accurate interpretations in earlier milestones. Fine-tuning provides incremental value by enforcing consistency and structure, but it does not fundamentally enhance the model’s ability to interpret experiment data. In practice, a combined approach that uses structured prompts with optional fine-tuning would provide the most effective balance between accuracy, control, and implementation effort.

From a practical perspective, this experiment demonstrates that fine-tuning is most valuable when consistent output structure and control are required, rather than when improving core analytical capability. In this case, the underlying model already possessed sufficient reasoning ability, and fine-tuning primarily served as a mechanism for enforcing predictable and repeatable responses.